In [1]:
import zarr
import xarray as xr
import metpy.calc as mpcalc
from metpy.units import units
import numpy as np
import pandas as pd

zarr_path = "/lustre/desc1/scratch/myasears/sounding_data/zarr/m2hats"
store = zarr.open(zarr_path, mode="r")
group_keys = list(store.group_keys())

### LCL calculation example

In [3]:
def compute_lcl_for_group(store_path, group_key):
    g = zarr.open_group(store_path, mode="r")[group_key]
    p = g["pres"][:] * units.hPa
    T = g["tdry"][:] * units.celsius
    Td = g["dp"][:] * units.celsius

    mask = np.isfinite(p) & np.isfinite(T) & np.isfinite(Td)
    if mask.sum() == 0:
        return group_key, np.nan, np.nan

    idx = np.argmax(p[mask].magnitude)  # surface = max pressure
    p0 = p[mask][idx]
    T0 = T[mask][idx]
    Td0 = Td[mask][idx]

    lcl_p, lcl_T = mpcalc.lcl(p0, T0, Td0)

    return group_key, lcl_p.magnitude, lcl_T.magnitude

### Two execution methods

In [4]:
# Method 1

from concurrent.futures import ProcessPoolExecutor, as_completed

results = []

with ProcessPoolExecutor() as exe:
    futures = {
        exe.submit(compute_lcl_for_group, zarr_path, key): key
        for key in group_keys
    }

    for fut in as_completed(futures):
        results.append(fut.result())

df = pd.DataFrame(results, columns=["sounding_id", "lcl_pressure_hpa", "lcl_temperature_K"])

In [10]:
# Method 2

from dask import delayed, compute

tasks = [delayed(compute_lcl_for_group)(zarr_path, key) for key in group_keys]
results = compute(*tasks)
df = (pd.DataFrame(results, columns=["sounding_id", "lcl_pressure_hpa", "lcl_temperature_K"])